In [1]:
# Polariton Disorder — Computation Notebook

# Computes the disorder self-energy on a uniform momentum mesh and
# reconstructs the on-shell Q(k, eta) by sweeping an external energy E_ext.
# Results are saved to `Results/` as `.npy` arrays with `_meta.json` sidecars.

# **Workflow**
# 1. Define a base `Params` and any sweep axes in *Cell 2 — Parameters & sweep*.
# 2. Run *Cell 3 — Sweep helpers* to expand the parameter combinations.
# 3. Run *Cell 4 — Kernel mesh* to build and save K(q,k) on the uniform mesh.
# 4. Run *Cell 5 — Sigma sweep* to compute Sigma(k, E_ext, eta) and reduce
#    it to on-shell Q(k, eta) via the root of Re[E_ext - bare(k) - Sigma].
# 5. Run *Cell 6 — Diagnostic* to plot E_k'(k) for each eta.
# 6. Open `Visualisations.ipynb` to plot saved Q results.


In [2]:
import sys
sys.path.insert(0, '..')

import numpy as np
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
from polaritons.parameters import Params, DEFAULT_PARAMS

# ---------------------------------------------------------------------------
# Base parameter set  (modify here to change the physics)
# ---------------------------------------------------------------------------

base = Params(
	# Material (GaAs)
	E_bind       = 4.2e-3,          # eV  -- exciton binding energy
	E_gap_bare   = 1.519,             # eV  -- bare band gap
	m_e          = 3.692e-13,        # eV s^2/m^2
	m_h          = 9.372e-13,        # eV s^2/m^2
	m_rest       = 5.68e-12,        # eV s^2/m^2

	# Polariton / cavity
	Omega        = 1.4e-2,          # eV  -- Rabi splitting
	m_prime      = 0.25,
	n_refr       = 3.0,             # cavity refractive index
	N_qw         = 1,               # number of quantum wells

	# Disorder
	D_0          = 2.26e-20,        # eV^2 m^2
	xi           = 20e-9,           # m  -- correlation length

	# Thermodynamics
	T            = 20.0,            # K
	concentration = 0.3e12,         # m^-2
	g_ex         = 12e-18,          # eV m^2
)

# ---------------------------------------------------------------------------
# Parameter sweep axes
#
# Each entry is  (param_name, list_of_values).
# All combinations are explored.  Use a single-element list to hold a param fixed.
# ---------------------------------------------------------------------------

SWEEP = {
	"xi"    : [20e-9],   # correlation length (m)
	"D_0"   : [2.26e-20],              # disorder strength -- single value keeps it fixed
}

# eta grid -- disorder amplitude sweep (always included)
eta_grid = np.round(np.linspace(0.0, 2.0, 3), 9)  # 21

print(f"Base params:\n  E_bind={base.E_bind*1e3:.1f} meV,  xi={base.xi*1e9:.0f} nm,  "
	f"T={base.T:.0f} K,  D_0={base.D_0:.2e} eV^2m^2,  m_prime={base.m_prime:g}")
print(f"Sweep axes: {list(SWEEP.keys())}")
print(f"eta_grid: {len(eta_grid)} points from {eta_grid[0]} to {eta_grid[-1]}")


Base params:
  E_bind=4.2 meV,  xi=20 nm,  T=20 K,  D_0=2.26e-20 eV^2m^2,  m_prime=0.25
Sweep axes: ['xi', 'D_0']
eta_grid: 3 points from 0.0 to 2.0


In [3]:
import itertools
from dataclasses import replace

def build_sweep_params(base: Params, sweep: dict) -> list[Params]:
	"""
	Return a list of Params objects, one for each combination of sweep values.

	Parameters
	----------
	base   : base Params (SI units)
	sweep  : dict mapping field names to lists of values to sweep over

	Example
	-------
	sweep = {"xi": [10e-9, 20e-9], "T": [10.0, 20.0]}
	→ 4 Params objects covering all (xi, T) combinations
	"""
	keys   = list(sweep.keys())
	values = list(sweep.values())
	combos = list(itertools.product(*values))

	params_list = []
	for combo in combos:
		overrides = dict(zip(keys, combo))
		p = replace(base, **overrides)
		params_list.append(p)
	return params_list


sweep_params_si      = build_sweep_params(base, SWEEP)
sweep_params_natural = [p.to_natural() for p in sweep_params_si]

print(f"{len(sweep_params_si)} parameter set(s) in sweep:")
for i, p in enumerate(sweep_params_si):
	print(f"  [{i}]  xi={p.xi*1e9:.0f} nm  D_0={p.D_0:.2e}  T={p.T:.0f} K")


1 parameter set(s) in sweep:
  [0]  xi=20 nm  D_0=2.26e-20  T=20 K


In [ ]:
from polaritons.kernel import make_kernel_gaussian, make_kernel_nongaussian, find_kernel_truncation
from polaritons.grid   import uniform_grid_and_weights
from polaritons.io     import save_result, make_sweep_stem
from polaritons.units  import k_cm_to_nat, k_nat_to_cm

# ---------------------------------------------------------------------------
# Grid configuration (per-kernel)
# ---------------------------------------------------------------------------

SWEEP_SCHEMA = "sigma_uniform_v1"

KERNEL_FACTORIES = {
	"nongaussian": make_kernel_nongaussian,
	"gaussian"   : make_kernel_gaussian,
}

# Truncation threshold: pick t per-kernel so that |K(t,t)| <= KERNEL_THRESHOLD * |K(0,0)|,
# lower-bounded by K_MIN_CM (cm^-1). Each kernel (non-Gaussian shared; Gaussian
# per ξ) uses its own t_local, and builds its own Picard grid on [0, t_local].
KERNEL_THRESHOLD = 0.05
K_MIN_CM         = 40_000.0      # lower bound on truncation in cm^-1
N_PICARD         = 7500         # number of uniform grid points

# Angular-quadrature nodes per kernel.  The Gaussian kernel develops a
# narrow peak in theta of width ~1/(xi*t) at large q,k; the trapezoid rule
# must oversample that peak, or its angular integral aliases to a numerical
# floor and the truncation search plateaus above KERNEL_THRESHOLD.
# n_gauss = 1024 keeps ~3x oversampling for xi*t up to ~200.
N_GAUSS_BY_KERNEL = {"gaussian": 1024, "nongaussian": 96}

kernel_jobs = []


def kernel_run_label(kernel_type, p_si, xi_independent=False):
	parts = [kernel_type]
	if not xi_independent:
		parts.append(f"ξ={p_si.xi*1e9:.0f} nm")
	parts.append(f"D_0={p_si.D_0:.2e}")
	parts.append(f"m_prime={p_si.m_prime:g}")
	return ", ".join(parts)


# ---------------------------------------------------------------------------
# Step 1: enumerate kernel jobs (non-Gaussian shared; Gaussian per ξ).
# ---------------------------------------------------------------------------
pending_jobs = []
pending_jobs.append(dict(
	p_si=base, p_nat=base.to_natural(), kernel_type="nongaussian",
	sweep_index=None, xi_independent=True,
))
for idx, (p_si, p_nat) in enumerate(zip(sweep_params_si, sweep_params_natural)):
	pending_jobs.append(dict(
		p_si=p_si, p_nat=p_nat, kernel_type="gaussian",
		sweep_index=idx, xi_independent=False,
	))

# ---------------------------------------------------------------------------
# Step 2: discover each kernel's per-kernel truncation t_local.
# ---------------------------------------------------------------------------
print("Discovering per-kernel truncations...")
for job in pending_jobs:
	p_nat       = job["p_nat"]
	K_fn        = KERNEL_FACTORIES[job["kernel_type"]](p_nat, n_gauss=N_GAUSS_BY_KERNEL[job["kernel_type"]])
	k_lower_nat = float(k_cm_to_nat(K_MIN_CM, p_nat))
	t_nat       = find_kernel_truncation(K_fn, threshold=KERNEL_THRESHOLD, k_lower=k_lower_nat)
	t_cm        = float(k_nat_to_cm(t_nat, p_nat))
	job["K_fn"]        = K_fn
	job["t_nat_local"] = t_nat
	job["t_cm_local"]  = t_cm
	job["k_lower_nat"] = k_lower_nat
	print(f"  {kernel_run_label(job['kernel_type'], job['p_si'], job['xi_independent'])}: "
	    f"t_local = {t_nat:.4g} (nat) = {t_cm:.3g} cm^-1")

per_job_t_nat = {
	f"{job['kernel_type']}|sweep={job['sweep_index']}": float(job["t_nat_local"])
	for job in pending_jobs
}

# ---------------------------------------------------------------------------
# Step 3: build a per-kernel grid and compute each kernel mesh on its own grid.
# ---------------------------------------------------------------------------
for job in pending_jobs:
	p_si           = job["p_si"]
	p_nat          = job["p_nat"]
	kernel_type    = job["kernel_type"]
	sweep_index    = job["sweep_index"]
	xi_independent = job["xi_independent"]
	K_fn           = job["K_fn"]
	t_nat_local    = job["t_nat_local"]
	print(f"\n-- {kernel_run_label(kernel_type, p_si, xi_independent)} --")

	q_picard, weights_picard = uniform_grid_and_weights(t_nat_local, N_PICARD)
	print(f"  Grid: N={N_PICARD}, K_max={t_nat_local:.4g}, dq={q_picard[1]-q_picard[0]:.4g}")
	print(f"  Computing kernel mesh ({N_PICARD}x{N_PICARD}) on this kernel's grid...")
	K_mesh = K_fn(q_picard, q_picard)

	extra_meta = {
		"kernel_type"        : kernel_type,
		"xi_independent"     : bool(xi_independent),
		"N_picard"           : N_PICARD,
		"K_domain_max"       : float(t_nat_local),
		"K_domain_max_local" : float(t_nat_local),
		"K_domain_max_shared": False,
		"per_job_t_nat"      : per_job_t_nat,
		"kernel_threshold"   : KERNEL_THRESHOLD,
		"k_min_cm"           : K_MIN_CM,
		"k_min_nat"          : job["k_lower_nat"],
		"q_picard"           : q_picard.tolist(),
		"sweep_index"        : sweep_index,
		"xi_m"               : None if xi_independent else p_si.xi,
		"m_prime"            : p_si.m_prime,
		"calculation_units"  : "natural",
		"sweep_schema"       : SWEEP_SCHEMA,
	}
	stem = make_sweep_stem("K", p_nat, extra={
		"kernel_type"    : kernel_type,
		"xi_independent" : bool(xi_independent),
		"sweep_schema"   : SWEEP_SCHEMA,
	})
	save_result(K_mesh, "Results/integrand_meshes", stem, p_nat, extra_meta)
	kernel_jobs.append({
		"sweep_index"    : sweep_index,
		"kernel_type"    : kernel_type,
		"xi_independent" : bool(xi_independent),
		"stem"           : stem,
		"p_si"           : p_si,
		"p_nat"          : p_nat,
		"K_domain_max"   : float(t_nat_local),
		"q_picard"       : q_picard,
		"weights_picard" : weights_picard,
	})

print(f"\nAll kernels saved.  Jobs: {[(job['kernel_type'], job['stem']) for job in kernel_jobs]}")


Discovering per-kernel truncations...
  nongaussian, D_0=2.26e-20, m_prime=0.25: t_local = 12.69 (nat) = 9.09e+06 cm^-1
  gaussian, ξ=20 nm, D_0=2.26e-20, m_prime=0.25: t_local = 4.444 (nat) = 3.19e+06 cm^-1


In [5]:
from polaritons.sigma import sweep_sigma, find_E_k_prime, assemble_Q
from polaritons.io    import load_result, save_result, make_sweep_stem

# ---------------------------------------------------------------------------
# Sigma sweep settings
# ---------------------------------------------------------------------------

PICARD_TOL      = 1e-6
PICARD_MAX_ITER = 15_000
PICARD_W        = 0.99
PICARD_VERBOSE  = True

# External-energy window. We solve one Picard per (η, E_ext) and reduce to
# the on-shell root E_k′ of  E_ext − bare(k) − Re[Σ] = 0. The root for k in
# [0, t_local] lies between roughly  -|Re[Σ(k=0)]|  and  bare(t_local) + |Re[Σ]|,
# so we size the window as
#     [-margin * Ω,  bare(t_local) + margin * Ω].
#
# Uniform sampling wastes resolution in the wings: the on-shell root sits on
# the bare parabola bare(k) = ħ²k²/(2M), so resolving Re[Σ] there to high
# precision is what fixes the small-Re[Q(0)] non-monotonicity. We therefore
# build a piecewise grid with E_EXT_BAND_FRACTION of all points crammed into
# the band region [E_band_lo, E_band_hi] = [-Σ_pad, bare_max + Σ_pad] and the
# remaining points spread linearly across each wing. The band pad Σ_pad is
# sized so that the dense region also covers the expected magnitude of
# Re[Σ(k)] above and below the bare parabola.
E_EXT_MARGIN_OMEGA   = 5.0
E_EXT_BAND_FRACTION  = 0.7   # fraction of points placed in the dense region
E_EXT_BAND_PAD_OMEGA = 0.5   # extra Ω-width around the bare parabola for the dense band
N_E_EXT              = 1501

def _default_E_ext_window(p_nat, t_local, margin_omega=E_EXT_MARGIN_OMEGA):
	bare_max = float(p_nat.hbar**2 * t_local**2 / (2.0 * p_nat.M))
	E_min    = -margin_omega * p_nat.Omega
	E_max    = bare_max + margin_omega * p_nat.Omega
	return E_min, E_max, bare_max


def _concentrated_E_ext_grid(
	p_nat,
	t_local,
	*,
	n              = N_E_EXT,
	margin_omega   = E_EXT_MARGIN_OMEGA,
	band_fraction  = E_EXT_BAND_FRACTION,
	band_pad_omega = E_EXT_BAND_PAD_OMEGA,
):
	"""Strictly-increasing E_ext grid concentrated around the bare parabola.

	The on-shell root sits at E_ext ≈ bare(k) + Re[Σ] for each k. Loading
	band_fraction of the N samples into the band [-pad, bare_max + pad]
	shrinks the spacing there from (E_max - E_min)/N to ≈ band_width/(band_fraction·N),
	directly improving the accuracy of the linear-bracket root used in
	`find_E_k_prime` without changing the window the sweep covers.
	"""
	E_min, E_max, bare_max = _default_E_ext_window(p_nat, t_local, margin_omega=margin_omega)
	pad        = band_pad_omega * p_nat.Omega
	E_band_lo  = max(E_min, -pad)
	E_band_hi  = min(E_max, bare_max + pad)
	if E_band_hi <= E_band_lo:
		# Degenerate band (shouldn't happen with sensible parameters); fall
		# back to a uniform grid so we still produce a usable result.
		return np.linspace(E_min, E_max, n)

	band_width = E_band_hi - E_band_lo
	wing_lo_w  = max(0.0, E_band_lo - E_min)
	wing_hi_w  = max(0.0, E_max - E_band_hi)
	wings_w    = wing_lo_w + wing_hi_w

	n_band  = max(2, int(round(band_fraction * n)))
	n_wings = max(0, n - n_band)
	if wings_w > 0.0:
		n_lo = max(2, int(round(n_wings * (wing_lo_w / wings_w)))) if wing_lo_w > 0.0 else 0
		n_hi = max(2, n_wings - n_lo) if wing_hi_w > 0.0 else 0
	else:
		n_lo = n_hi = 0

	# Endpoint handling: each segment is built to include both endpoints, then
	# concatenated and de-duplicated so the join points appear exactly once.
	pieces = []
	if n_lo > 0:
		pieces.append(np.linspace(E_min, E_band_lo, n_lo))
	pieces.append(np.linspace(E_band_lo, E_band_hi, n_band))
	if n_hi > 0:
		pieces.append(np.linspace(E_band_hi, E_max, n_hi))

	grid = np.unique(np.concatenate(pieces))
	if grid.size < n:
		# `np.unique` removed the shared join points; top up with extra
		# uniform points inside the dense band to keep the total ≈ N.
		extra_n = n - grid.size
		extra   = np.linspace(E_band_lo, E_band_hi, n_band + extra_n + 2)[1:-1]
		grid    = np.unique(np.concatenate([grid, extra]))
	return grid

SAVE_SIGMA = True

for job in kernel_jobs:
	kernel_type    = job["kernel_type"]
	k_stem         = job["stem"]
	p_si           = job["p_si"]
	p_nat          = job["p_nat"]
	sweep_index    = job["sweep_index"]
	xi_independent = job["xi_independent"]
	print(f"\n-- Sigma sweep, {kernel_run_label(kernel_type, p_si, xi_independent)} --")

	K_mesh, k_meta = load_result("Results/integrand_meshes", k_stem)
	q            = np.array(k_meta["q_picard"])
	N_pic        = len(q)
	K_domain_max = float(k_meta["K_domain_max"])
	weights      = uniform_grid_and_weights(K_domain_max, N_pic)[1]

	E_ext_grid = _concentrated_E_ext_grid(p_nat, float(k_meta["K_domain_max"]))
	diffs      = np.diff(E_ext_grid)
	bare_max_j = float(p_nat.hbar**2 * float(k_meta["K_domain_max"])**2 / (2.0 * p_nat.M))
	print(f"  E_ext window: [{E_ext_grid[0]:.4g}, {E_ext_grid[-1]:.4g}] "
	      f"(natural energy units), N={E_ext_grid.size}")
	print(f"  E_ext spacing: min={diffs.min():.3e}, max={diffs.max():.3e} "
	      f"(dense band centred on bare_max={bare_max_j:.3g})")
	print(f"  eta grid    : {len(eta_grid)} points")

	Sigma_arr, iters = sweep_sigma(
		p_nat, K_mesh, q, weights, E_ext_grid, eta_grid,
		tol=PICARD_TOL, max_iter=PICARD_MAX_ITER, w=PICARD_W,
		verbose=PICARD_VERBOSE,
	)
	print(f"  Picard iters: mean={iters.mean():.1f}, max={iters.max()}, "
	      f"hit max_iter on {(iters == PICARD_MAX_ITER).sum()} solves")

	E_k_prime = find_E_k_prime(Sigma_arr, q, E_ext_grid, p_nat)
	Q_results = assemble_Q(Sigma_arr, E_k_prime, E_ext_grid)

	nan_per_eta = np.isnan(E_k_prime).sum(axis=1)
	if nan_per_eta.any():
		print(f"  WARN: NaN E_k' counts per eta: {nan_per_eta.tolist()} "
		      "(consider widening E_ext)")

	extra_q = {
		"eta_grid"              : eta_grid.tolist(),
		"q_picard"              : q.tolist(),
		"E_ext_grid"            : E_ext_grid.tolist(),
		"E_ext_grid_kind"       : "concentrated_bare_parabola",
		"E_ext_band_fraction"   : E_EXT_BAND_FRACTION,
		"E_ext_band_pad_omega"  : E_EXT_BAND_PAD_OMEGA,
		"E_ext_margin_omega"    : E_EXT_MARGIN_OMEGA,
		"kernel_stem"           : k_stem,
		"kernel_type"           : kernel_type,
		"xi_independent"        : bool(xi_independent),
		"K_domain_max"          : K_domain_max,
		"kernel_threshold"      : float(k_meta.get("kernel_threshold", 0.0)) or None,
		"k_min_cm"              : float(k_meta.get("k_min_cm", 0.0)) or None,
		"picard_tol"            : PICARD_TOL,
		"picard_max_iter"       : PICARD_MAX_ITER,
		"picard_w"              : PICARD_W,
		"sweep_index"           : sweep_index,
		"xi_m"                  : None if xi_independent else p_si.xi,
		"m_prime"               : p_si.m_prime,
		"calculation_units"     : "natural",
		"sweep_schema"          : SWEEP_SCHEMA,
		"saved_sigma"           : bool(SAVE_SIGMA),
		"convention"            : "band_bottom_relative",
	}
	q_stem = make_sweep_stem("Q", p_nat, extra={
		"kernel_stem"    : k_stem,
		"kernel_type"    : kernel_type,
		"xi_independent" : bool(xi_independent),
		"sweep_schema"   : SWEEP_SCHEMA,
	})
	save_result(Q_results, "Results/Q_results", q_stem, p_nat, extra_q)

	if SAVE_SIGMA:
		import os
		base_dir = "Results/Q_results"
		np.save(os.path.join(base_dir, f"{q_stem}_Sigma.npy"),       Sigma_arr)
		np.save(os.path.join(base_dir, f"{q_stem}_E_k_prime.npy"),   E_k_prime)
		np.save(os.path.join(base_dir, f"{q_stem}_E_ext_grid.npy"),  E_ext_grid)
		np.save(os.path.join(base_dir, f"{q_stem}_iters.npy"),       iters)

	job["q"]         = q
	job["E_ext"]     = E_ext_grid
	job["Sigma"]     = Sigma_arr
	job["E_k_prime"] = E_k_prime
	job["Q"]         = Q_results

print("\nAll Sigma sweeps complete.")



-- Sigma sweep, nongaussian, D_0=2.26e-20, m_prime=0.25 --
  E_ext window: [-16.67, 49.29] (natural energy units), N=2551
  E_ext spacing: min=6.499e-05, max=6.696e-02 (dense band centred on bare_max=32.6)
  eta grid    : 3 points
  [eta=1 E_ext=-16.67] iter     0  |ΔQ| = 2.368e+01
  [eta=1 E_ext=-16.6] iter     0  |ΔQ| = 7.587e-02
  [eta=1 E_ext=-16.53] iter     0  |ΔQ| = 7.639e-02
  [eta=1 E_ext=-16.47] iter     0  |ΔQ| = 7.692e-02
  [eta=1 E_ext=-16.4] iter     0  |ΔQ| = 7.745e-02
  [eta=1 E_ext=-16.33] iter     0  |ΔQ| = 7.799e-02
  [eta=1 E_ext=-16.26] iter     0  |ΔQ| = 7.854e-02
  [eta=1 E_ext=-16.2] iter     0  |ΔQ| = 7.909e-02
  [eta=1 E_ext=-16.13] iter     0  |ΔQ| = 7.964e-02
  [eta=1 E_ext=-16.06] iter     0  |ΔQ| = 8.021e-02
  [eta=1 E_ext=-16] iter     0  |ΔQ| = 8.078e-02
  [eta=1 E_ext=-15.93] iter     0  |ΔQ| = 8.135e-02
  [eta=1 E_ext=-15.86] iter     0  |ΔQ| = 8.193e-02
  [eta=1 E_ext=-15.8] iter     0  |ΔQ| = 8.252e-02
  [eta=1 E_ext=-15.73] iter     0  |ΔQ| = 8.312

In [ ]:
# ---------------------------------------------------------------------------
# Cell 5 — Sigma reuse (resume / rerun / refine a prior surface)
# ---------------------------------------------------------------------------
#
# Set REUSE_MODE to one of:
#   None      : skip this cell (default)
#   "resume"  : redo only the cells that did not converge in the previous run,
#               using a stronger damping factor (smaller PICARD_W_NEW)
#   "rerun"   : redo every cell starting from the previous surface, e.g. with
#               a weaker damping factor (larger PICARD_W_NEW) to refine it
#   "refine"  : add N_E_EXT_ADD extra E_ext samples around the bare parabola
#               band and solve Picard only at the new columns; old columns
#               are copied through and the on-shell Q is recomputed on the
#               denser grid.
#
# All modes require PREV_Q_STEM (the q_stem string of the prior run, e.g.
# "Q_13d8e8") and reuse the same kernel/momentum grid that is already loaded
# into `kernel_jobs` above. The kernel_stem stored in the loaded meta is
# verified against the current job to catch mismatches.

REUSE_MODE              = "resume"        # None | "resume" | "rerun" | "refine"
PREV_Q_STEM             = "Q_13d8e8"          # e.g. "Q_13d8e8"
PICARD_W_NEW            = 0.95         # damping for resume/rerun (smaller = stronger damping)
PICARD_MAX_ITER_NEW     = 30_000      # iteration budget for resume/rerun/refine
PICARD_TOL_NEW          = 1e-6          # tolerance for resume/rerun/refine
N_E_EXT_ADD             = 200         # extra E_ext points for "refine"
E_EXT_REFINE_PAD_OMEGA  = 0.5         # band pad (in Omega) for "refine"

if REUSE_MODE is not None:
        from polaritons.sigma import (
                resume_unconverged, rerun_with_seed, refine_sigma,
        )
        if REUSE_MODE not in {"resume", "rerun", "refine"}:
                raise ValueError(f"Unknown REUSE_MODE: {REUSE_MODE!r}")
        if not PREV_Q_STEM:
                raise ValueError("PREV_Q_STEM must be set when REUSE_MODE is not None.")

        for job in kernel_jobs:
                kernel_type    = job["kernel_type"]
                k_stem         = job["stem"]
                p_si           = job["p_si"]
                p_nat          = job["p_nat"]
                sweep_index    = job["sweep_index"]
                xi_independent = job["xi_independent"]
                print(f"\n-- Sigma reuse ({REUSE_MODE}), "
                      f"{kernel_run_label(kernel_type, p_si, xi_independent)} --")

                base_dir = "Results/Q_results"
                Sigma_prev   = np.load(os.path.join(base_dir, f"{PREV_Q_STEM}_Sigma.npy"))
                E_ext_prev   = np.load(os.path.join(base_dir, f"{PREV_Q_STEM}_E_ext_grid.npy"))
                iters_prev_p = os.path.join(base_dir, f"{PREV_Q_STEM}_iters.npy")
                iters_prev   = np.load(iters_prev_p) if os.path.exists(iters_prev_p) else None
                _, prev_meta = load_result(base_dir, PREV_Q_STEM)

                if prev_meta.get("kernel_stem") != k_stem:
                        raise RuntimeError(
                                f"Previous run was computed with kernel_stem="
                                f"{prev_meta.get('kernel_stem')!r}, but current "
                                f"job has kernel_stem={k_stem!r}. Aborting."
                        )

                K_mesh, k_meta = load_result("Results/integrand_meshes", k_stem)
                q            = np.array(k_meta["q_picard"])
                N_pic        = len(q)
                K_domain_max = float(k_meta["K_domain_max"])
                weights      = uniform_grid_and_weights(K_domain_max, N_pic)[1]

                if REUSE_MODE == "resume":
                        prev_max_iter = int(prev_meta.get("picard_max_iter", PICARD_MAX_ITER))
                        if iters_prev is None:
                                raise RuntimeError(
                                        "resume mode needs <stem>_iters.npy from the previous "
                                        "run; not found on disk."
                                )
                        Sigma_arr, iters = resume_unconverged(
                                p_nat, K_mesh, q, weights, E_ext_prev, eta_grid,
                                Sigma_prev, iters_prev, prev_max_iter,
                                w=PICARD_W_NEW, max_iter=PICARD_MAX_ITER_NEW,
                                tol=PICARD_TOL_NEW, verbose=PICARD_VERBOSE,
                        )
                        E_ext_grid_out = E_ext_prev

                elif REUSE_MODE == "rerun":
                        Sigma_arr, iters = rerun_with_seed(
                                p_nat, K_mesh, q, weights, E_ext_prev, eta_grid,
                                Sigma_prev,
                                w=PICARD_W_NEW, max_iter=PICARD_MAX_ITER_NEW,
                                tol=PICARD_TOL_NEW, verbose=PICARD_VERBOSE,
                        )
                        E_ext_grid_out = E_ext_prev

                else:  # "refine"
                        Sigma_arr, iters, E_ext_grid_out = refine_sigma(
                                p_nat, K_mesh, q, weights, E_ext_prev, eta_grid,
                                Sigma_prev, t_local=K_domain_max,
                                n_add=N_E_EXT_ADD,
                                band_pad_omega=E_EXT_REFINE_PAD_OMEGA,
                                w=PICARD_W_NEW, max_iter=PICARD_MAX_ITER_NEW,
                                tol=PICARD_TOL_NEW, verbose=PICARD_VERBOSE,
                        )

                print(f"  Picard iters: mean={iters.mean():.1f}, max={iters.max()}, "
                      f"hit max_iter on {(iters == PICARD_MAX_ITER_NEW).sum()} solves")

                E_k_prime = find_E_k_prime(Sigma_arr, q, E_ext_grid_out, p_nat)
                Q_results = assemble_Q(Sigma_arr, E_k_prime, E_ext_grid_out)

                nan_per_eta = np.isnan(E_k_prime).sum(axis=1)
                if nan_per_eta.any():
                        print(f"  WARN: NaN E_k' counts per eta: {nan_per_eta.tolist()}")

                extra_q = {
                        "eta_grid"             : eta_grid.tolist(),
                        "q_picard"             : q.tolist(),
                        "E_ext_grid"           : E_ext_grid_out.tolist(),
                        "E_ext_grid_kind"      : prev_meta.get("E_ext_grid_kind", "unknown")
                                                 + (f"+refined+{N_E_EXT_ADD}" if REUSE_MODE == "refine" else ""),
                        "kernel_stem"          : k_stem,
                        "kernel_type"          : kernel_type,
                        "xi_independent"       : bool(xi_independent),
                        "K_domain_max"         : K_domain_max,
                        "picard_tol"           : PICARD_TOL_NEW,
                        "picard_max_iter"      : PICARD_MAX_ITER_NEW,
                        "picard_w"             : PICARD_W_NEW,
                        "sweep_index"          : sweep_index,
                        "xi_m"                 : None if xi_independent else p_si.xi,
                        "m_prime"              : p_si.m_prime,
                        "calculation_units"    : "natural",
                        "sweep_schema"         : SWEEP_SCHEMA,
                        "saved_sigma"          : True,
                        "convention"           : "band_bottom_relative",
                        "parent_stem"          : PREV_Q_STEM,
                        "reuse_mode"           : REUSE_MODE,
                }
                q_stem_new = make_sweep_stem("Q", p_nat, extra={
                        "kernel_stem"    : k_stem,
                        "kernel_type"    : kernel_type,
                        "xi_independent" : bool(xi_independent),
                        "sweep_schema"   : SWEEP_SCHEMA,
                        "parent_stem"    : PREV_Q_STEM,
                        "reuse_mode"     : REUSE_MODE,
                })
                save_result(Q_results, base_dir, q_stem_new, p_nat, extra_q)
                np.save(os.path.join(base_dir, f"{q_stem_new}_Sigma.npy"),       Sigma_arr)
                np.save(os.path.join(base_dir, f"{q_stem_new}_E_k_prime.npy"),   E_k_prime)
                np.save(os.path.join(base_dir, f"{q_stem_new}_E_ext_grid.npy"),  E_ext_grid_out)
                np.save(os.path.join(base_dir, f"{q_stem_new}_iters.npy"),       iters)
                print(f"  saved as {q_stem_new}")

                job["q"]         = q
                job["E_ext"]     = E_ext_grid_out
                job["Sigma"]     = Sigma_arr
                job["E_k_prime"] = E_k_prime
                job["Q"]         = Q_results

        print(f"\nReuse pass ({REUSE_MODE}) complete.")
else:
        print("REUSE_MODE is None — skipping reuse pass.")
